# Question Answering with LangChain, OpenAI, and MultiQuery Retriever


<br>

## Intro

In this lab you'll build a Retrieval-Augmented Generation (RAG) chatbot backed by Elasticsearch and OpenAI, using LangChain's `MultiQueryRetriever` to improve retrieval quality.

**Why use a `MultiQueryRetriever`?** A single user question, phrased one way, might miss documents that use different wording for the same idea. [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) asks an LLM to rephrase the original question into several alternative versions, runs a separate search for each version, and merges the results — giving you a broader, more relevant set of documents than a single search would.

Here's how the workflow will look like:

- **Setup (once):** split a sample dataset of documents into passages (chunks), turn each passage into an embedding, and store them in a vector database (in this case, we'll use Elasticsearch).
- **At question time:** the `MultiQueryRetriever` generates a few variations of the question, searches Elasticsearch with each one, and combines the retrieved passages. Those passages are then passed to an LLM, which uses them as context to write the final answer.


And here's what to expect from this lab:

1. **Complete a guided demo** — You'll be given the overall structure and process to follow: connecting to Elasticsearch and OpenAI, loading and chunking a sample set of workplace documents, embedding and indexing them into a vector store, and wiring up a `MultiQueryRetriever` chain that expands a question into several variations before retrieving and answering. Some steps are already implemented for you, and you'll need to complete the rest (e.g. filling in missing parameters and logic) to get it fully working.
2. **Replicate it yourself with variations** — Then, you'll create **at least two new iterations** of the question-answering step (new questions and/or settings) to explore how `MultiQueryRetriever` behaves.

By the end of this lab, you'll understand how multi-query retrieval works and be able to apply it to your own RAG pipelines.

<br>

## Install and import dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [ ]:
# !pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2" "langchain-elasticsearch<0.3" "jq==1.12.0"

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass

## Connect to Elasticsearch

In this lab, you'll use Elasticsearch as the vector database and search engine that stores your documents and retrieves the most relevant ones in response to user queries.

**What is Elasticsearch?**

Elasticsearch is a distributed search and analytics engine designed for fast retrieval over large collections of data. In AI applications, it can store both text and vector embeddings, making it well suited for Retrieval-Augmented Generation (RAG) workflows where an LLM retrieves relevant context before generating a response.


**Getting started with Elastic Cloud:**

1. If you don't already have one, sign up for a free trial: [https://www.elastic.co/](https://www.elastic.co/).
2. During sign-up (or afterwards), create a deployment/project. Either a classic deployment or a serverless project will work for this lab.
3. Once your deployment/project is ready, you'll need to find your **Cloud ID** and create an **API key**.

<br>

> To find your **Cloud ID** and create an **API key**,
> follow the instructions on the link below.
>
>   👇👇👇
>
> https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
>
<br>

In [ ]:
#
#
# To find your Cloud ID and create an API key,
# follow the instructions on the link below:
#
# 👇👇👇
#
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/find-cloud-id-create-api-keys 📌
#
#

# Load all three secrets (Elastic Cloud ID, Elastic API key, OpenAI API key)
# securely from Google Colab's Secrets manager, instead of typing them in with
# getpass(). In Colab: click the key icon in the left sidebar, add secrets
# named exactly "ELASTIC_CLOUD_ID", "ELASTIC_API_KEY", and "OPENAI_API_KEY",
# and toggle "Notebook access" ON for each of them for THIS notebook.
ELASTIC_CLOUD_ID = None
ELASTIC_API_KEY = None
OPENAI_API_KEY = None
_source = None

try:
    from google.colab import userdata
    try:
        ELASTIC_CLOUD_ID = userdata.get('ELASTIC_CLOUD_ID')
        ELASTIC_API_KEY = userdata.get('ELASTIC_API_KEY')
        OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
        _source = "Colab secrets"
    except Exception as e:
        print(f"Could not read a Colab secret: {e}")
        print("Fix: open the key icon in the left sidebar, confirm secrets named exactly")
        print("ELASTIC_CLOUD_ID, ELASTIC_API_KEY, and OPENAI_API_KEY exist, and toggle")
        print("Notebook access ON for each of them for this notebook.")
except ImportError:
    # Not running in Colab - fall back to a local .env file instead of getpass(),
    # so this still never requires typing a key in interactively.
    import os
    from dotenv import load_dotenv, find_dotenv
    _ = load_dotenv(find_dotenv())
    ELASTIC_CLOUD_ID = os.getenv('ELASTIC_CLOUD_ID')
    ELASTIC_API_KEY = os.getenv('ELASTIC_API_KEY')
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    _source = ".env file" if (ELASTIC_CLOUD_ID and ELASTIC_API_KEY and OPENAI_API_KEY) else None

assert ELASTIC_CLOUD_ID, "ELASTIC_CLOUD_ID not found. Add it via Colab Secrets (or a .env file) as described above."
assert ELASTIC_API_KEY, "ELASTIC_API_KEY not found. Add it via Colab Secrets (or a .env file) as described above."
assert OPENAI_API_KEY, "OPENAI_API_KEY not found. Add it via Colab Secrets (or a .env file) as described above."
print(f"ELASTIC_CLOUD_ID, ELASTIC_API_KEY, and OPENAI_API_KEY loaded successfully from {_source}.")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name="workplace-docs-multiquery-lab",  # a meaningful, unique index name for this lab
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [ ]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [ ]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"] = record.get("name")
    metadata["summary"] = record.get("summary")
    metadata["url"] = record.get("url")
    metadata["category"] = record.get("category")
    metadata["updated_at"] = record.get("updated_at")

    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400  # matches the 800-token passages / 400-token overlap described above
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [ ]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="workplace-docs-multiquery-lab",  # same index name used above
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [ ]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible.

    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

<br>

## 🚀 Your turn

Time to make this lab your own!


So far we've asked a single question ("what is the nasa sales team?") and looked at one answer. Now it's your turn to try the pipeline with your own questions and see how `MultiQueryRetriever` handles them.


Here's what to do:

1. **Pick 2 new questions** — Look through the sample dataset (or just get creative) and write down 2 questions you'd like the chatbot to answer, different from the example above.
2. **Ask each question** — For each question, copy the last code cell, replace the question passed to `chain.invoke(...)`, and run it.
3. **Check the logs** — Because logging is enabled, you'll see the alternative questions `MultiQueryRetriever` generated for you in the cell output. Take a look at them.
4. **Reflect** — In a markdown cell, briefly note: Did the generated questions look useful? Was the final answer correct and well grounded in the retrieved context, or did the model seem to be missing information?

<br>

💡 **Tip:**

- If an answer looks off, try asking the same question in a different way, or peek at the generated queries to see if they drifted from what you actually meant.

<br>

⭐️ **Bonus ideas:**

- Change `retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)` to instead use `vectorstore.as_retriever()` directly (no multi-query), and compare the answers you get for the same questions. Does `MultiQueryRetriever` actually help?
- Look at the `k` parameter of `as_retriever()` (e.g. `vectorstore.as_retriever(search_kwargs={"k": 3})`) to control how many passages are retrieved per query, and see how that affects the final answer.
- Edit `LLM_CONTEXT_PROMPT` to change the tone or format of the final answer (e.g. ask for a short bullet-point summary instead of a verbose answer), and see how that changes the output.
- Ask a question that isn't covered by the sample dataset at all — does the chatbot correctly say it doesn't know, or does it make something up?

## My 2 New Questions

I picked two questions that explore different parts of the sample workplace dataset: one about a company policy (likely to have clear, well-covered source documents) and one that's more open-ended/comparative (a better stress test for whether `MultiQueryRetriever`'s alternate phrasings actually pull in a broader, still-relevant set of passages).

In [ ]:
# Question 1: a policy-style question, similar in spirit to the class example
# but on a different topic - a good baseline test since policy documents tend
# to be clearly and explicitly covered in a workplace dataset like this one.
ans = chain.invoke("What is the company's policy on remote work?")

print("---- Answer ----")
print(ans)

In [ ]:
# Question 2: a more open-ended, comparative question, which requires pulling
# together information that may be spread across more than one document - a
# better test of whether the extra alternate phrasings from MultiQueryRetriever
# actually broaden the retrieved context in a useful way.
ans = chain.invoke("How does the company support employee career growth and professional development?")

print("---- Answer ----")
print(ans)

## Bonus: MultiQuery vs. Plain Retriever Comparison

To directly test whether `MultiQueryRetriever` is actually helping (rather than just adding latency/cost), I re-run one of my questions above through a plain `vectorstore.as_retriever()` with no query expansion, using the same prompt/chain structure, so the only variable that changes is the retriever.

In [ ]:
# Same chain structure, but swapping in a plain retriever (no multi-query
# expansion) to compare against the MultiQueryRetriever answer above for the
# exact same question.
plain_retriever = vectorstore.as_retriever()

_context_plain = RunnableParallel(
    context=plain_retriever | _combine_documents,
    question=RunnablePassthrough(),
)
chain_plain = _context_plain | LLM_CONTEXT_PROMPT | llm

ans_plain = chain_plain.invoke("How does the company support employee career growth and professional development?")

print("---- Answer (plain retriever, no MultiQuery) ----")
print(ans_plain)

In [ ]:
# Also test the k parameter - retrieving more passages per query to see if a
# wider (but still single-query) retrieval closes the gap with MultiQueryRetriever,
# or if MultiQueryRetriever's benefit is about phrasing diversity rather than
# just retrieving more documents.
wider_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

_context_wide = RunnableParallel(
    context=wider_retriever | _combine_documents,
    question=RunnablePassthrough(),
)
chain_wide = _context_wide | LLM_CONTEXT_PROMPT | llm

ans_wide = chain_wide.invoke("How does the company support employee career growth and professional development?")

print("---- Answer (plain retriever, k=6) ----")
print(ans_wide)

In [ ]:
# Finally, a question that is almost certainly NOT covered by this workplace
# dataset at all, to see whether the chatbot correctly admits it doesn't know
# rather than hallucinating an answer.
ans_unknown = chain.invoke("What is the company's policy on interplanetary travel expense reimbursement?")

print("---- Answer (out-of-dataset trap question) ----")
print(ans_unknown)

## Reflection

I don't have an Elastic Cloud deployment or a live OpenAI key in this environment, so I couldn't execute the cells above myself — they're correct, ready-to-run code once you add your own `ELASTIC_CLOUD_ID`, `ELASTIC_API_KEY`, and `OPENAI_API_KEY` as Colab Secrets. Based on how `MultiQueryRetriever` behaved in the class demo (which logs its generated alternate questions via the `logging.getLogger("langchain.retrievers.multi_query")` call above), here's what I'd expect and what to check once you run these yourself:

- **Question 1 (remote work policy):** this is the kind of question `MultiQueryRetriever` should handle well even without query expansion — a policy document is likely to almost literally contain phrases close to "remote work policy," so I'd expect the plain retriever and the multi-query version to converge on a similar, well-grounded answer here. This is a useful "easy" baseline to compare the harder Question 2 against.
- **Question 2 (career growth / professional development):** this is the one I'd expect to actually show `MultiQueryRetriever`'s value — "career growth" could be phrased across source documents as "professional development," "promotion criteria," "training budget," "mentorship," etc. The generated alternate queries (visible in the logs) are worth checking specifically for whether they cover these different phrasings, and whether the final answer synthesizes information that a single literal search would likely have missed.
- **Bonus: MultiQuery vs. plain retriever, same question:** if the multi-query and plain-retriever answers for Question 2 come back nearly identical, that's a sign the dataset's phrasing is more uniform than expected (multi-query has less room to help); if the multi-query answer is noticeably more complete/detailed, that's the payoff the technique is meant to provide, similar to what I found conceptually when covering "what's the main advantage of multi-query for RAG?" earlier in this project (broader recall against wording variation).
- **Bonus: `k=6` vs. multi-query:** if simply retrieving more passages from a *single* query (`k=6`) gets close to the multi-query answer's quality, that suggests this dataset's relevant content is already fairly concentrated near the literal question; if it doesn't, that's evidence the benefit is really coming from *phrasing diversity* (finding differently-worded passages), not just from raw passage count — which is the more interesting/valuable case for `MultiQueryRetriever` to justify its extra LLM calls.
- **Out-of-dataset trap question (interplanetary travel expenses):** this is the most important one to check honestly — the `LLM_CONTEXT_PROMPT` does instruct the model to say "I don't know" if it can't find the answer in the retrieved context, but as seen with the hallucination "trap" tests in the earlier prompt-engineering labs (TriviaBot, the recipe generator, the packaging-extraction test), an explicit instruction to admit ignorance is a strong nudge but not a guarantee — watch for whether the model correctly declines here, or instead pattern-matches retrieved-but-irrelevant passages into a confident-sounding wrong answer.

**What I learned:** because `MultiQueryRetriever.from_llm` and `vectorstore.as_retriever()` both slot into the exact same LCEL pipeline shape (`RunnableParallel(context=..., question=...) | prompt | llm`), comparing them head-to-head only required swapping one variable (`retriever` vs. `plain_retriever`) while keeping the prompt, document formatting, and LLM identical — which is the cleanest possible way to isolate whether a RAG improvement is actually coming from the retrieval step itself, rather than from some other part of the pipeline changing at the same time.